# Subtaller 2: Repaso de SQL con Python

En este taller usaremos `sqlite3` (incluido en Python) para practicar conceptos fundamentales de SQL sobre una base de datos de e-commerce.

## Modelo de datos
```
customers     products       orders           order_items
----------    ----------     ----------       -----------
customer_id   product_id     order_id         item_id
name          name           customer_id  ->  order_id
email         category       order_date       product_id
city          price          status           quantity
country       stock                           unit_price
```

---
## Contenido
1. Configuración de la base de datos
2. SELECT básico y filtros (WHERE)
3. Ordenamiento (ORDER BY) y límites (LIMIT)
4. Funciones de agregación (COUNT, SUM, AVG, MAX, MIN)
5. Agrupación (GROUP BY) y filtros de grupo (HAVING)
6. JOIN entre tablas
7. Subconsultas
8. Funciones de texto y fecha
9. Ejercicios integradores

## 0. Configuración del entorno

Sigue estos pasos **una sola vez** antes de ejecutar el notebook.

---

### Paso 1 — Instalar Python

Selecciona tu sistema operativo:

#### Windows
1. Descarga el instalador desde https://www.python.org/downloads/
2. Ejecuta el instalador y **marca la casilla "Add Python to PATH"** antes de continuar
3. Verifica abriendo **CMD** o **PowerShell**:
   ```
   python --version
   ```

#### macOS
Python 3 no viene instalado por defecto en versiones recientes. Opciones:

- **Opción A — Instalador oficial:** descarga desde https://www.python.org/downloads/
- **Opción B — Homebrew** (recomendado si ya lo tienes):
  ```
  brew install python
  ```
Verifica en la **Terminal**:
```
python3 --version
```

#### Linux (Ubuntu / Debian)
Python 3 suele venir preinstalado. Si no, ejecuta en la terminal:
```
sudo apt update && sudo apt install python3 python3-pip python3-venv
```
Verifica:
```
python3 --version
```

> En todos los casos se requiere **Python 3.10 o superior**.

---

### Paso 2 — Instalar VS Code

Si no tienes VS Code, descárgalo desde https://code.visualstudio.com/ y sigue el instalador para tu sistema operativo.

---

### Paso 3 — Instalar las extensiones de VS Code

1. Abre VS Code
2. Ve a **Extensiones** (`Ctrl+Shift+X` en Windows/Linux · `Cmd+Shift+X` en macOS)
3. Busca e instala:
   - **Python** (publicado por Microsoft)
   - **Jupyter** (publicado por Microsoft)

---

### Paso 4 — Seleccionar el intérprete (kernel)

1. Abre este archivo `.ipynb` en VS Code
2. En la esquina superior derecha aparece el botón **"Select Kernel"**
3. Haz clic → **"Python Environments..."** → elige la versión de Python que instalaste

> Si no aparece ningún intérprete, cierra y vuelve a abrir VS Code después de instalar Python.

---

### Paso 5 — Instalar dependencias

Ejecuta la celda de código a continuación (`Shift+Enter`).  
Instalará `pandas`; `sqlite3` ya viene incluido en Python y no requiere instalación.

In [ ]:
import subprocess, sys

# Instalar pandas si no está disponible
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas'])

print('Dependencias listas.')
print(f'  - sqlite3: incluido en Python {sys.version.split()[0]}')

import pandas as pd
print(f'  - pandas {pd.__version__}')

## 1. Configuración: Crear la base de datos

In [ ]:
import sqlite3
import pandas as pd

# Crear conexión en memoria (o cambiar ':memory:' por 'ecommerce.db' para persistir)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Función auxiliar para mostrar resultados como DataFrame
def query(sql, params=None):
    if params:
        return pd.read_sql_query(sql, conn, params=params)
    return pd.read_sql_query(sql, conn)

print('Conexión establecida exitosamente.')

In [ ]:
# Crear tablas
cursor.executescript('''
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    email       TEXT UNIQUE,
    city        TEXT,
    country     TEXT DEFAULT 'Colombia'
);

CREATE TABLE products (
    product_id  TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    category    TEXT,
    price       REAL NOT NULL,
    stock       INTEGER DEFAULT 0
);

CREATE TABLE orders (
    order_id    TEXT PRIMARY KEY,
    customer_id TEXT REFERENCES customers(customer_id),
    order_date  TEXT,
    status      TEXT CHECK(status IN ('entregado','enviado','cancelado','pendiente'))
);

CREATE TABLE order_items (
    item_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id   TEXT REFERENCES orders(order_id),
    product_id TEXT REFERENCES products(product_id),
    quantity   INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    discount   REAL DEFAULT 0.0
);
''')
conn.commit()
print('Tablas creadas.')

In [ ]:
# Insertar datos de ejemplo
customers_data = [
    ('C001','Ana García','ana.garcia@email.com','Bogotá','Colombia'),
    ('C002','Carlos López','carlos.lopez@email.com','Medellín','Colombia'),
    ('C003','María Rodríguez','maria.rodriguez@email.com','Cali','Colombia'),
    ('C004','Pedro Martínez','pedro.martinez@email.com','Barranquilla','Colombia'),
    ('C005','Laura Sánchez','laura.sanchez@email.com','Cartagena','Colombia'),
    ('C006','Juan Torres','juan.torres@email.com','Bogotá','Colombia'),
    ('C007','Sofía Herrera','sofia.herrera@email.com','Medellín','Colombia'),
    ('C008','Diego Flores','diego.flores@email.com','Cali','Colombia'),
    ('C009','Valentina Cruz','valentina.cruz@email.com','Bucaramanga','Colombia'),
    ('C010','Andrés Jiménez','andres.jimenez@email.com','Bogotá','Colombia'),
]

products_data = [
    ('P101','Laptop HP 15','Electrónica',1200.00,15),
    ('P205','Auriculares Sony WH','Electrónica',150.00,40),
    ('P310','Silla Ergonómica','Muebles',350.00,20),
    ('P410','Mouse Inalámbrico','Electrónica',45.00,100),
    ('P501','Libro Python Data Science','Libros',35.00,60),
    ('P602','Teclado Mecánico','Electrónica',120.00,30),
    ('P710','Cámara Web HD','Electrónica',80.00,25),
    ('P801','Monitor 24 pulgadas','Electrónica',280.00,18),
    ('P901','Escritorio de Pie','Muebles',500.00,10),
]

orders_data = [
    ('ORD-001','C001','2024-01-05','entregado'),
    ('ORD-002','C002','2024-01-06','entregado'),
    ('ORD-003','C003','2024-01-07','enviado'),
    ('ORD-004','C001','2024-01-08','entregado'),
    ('ORD-005','C004','2024-01-10','entregado'),
    ('ORD-006','C005','2024-01-12','cancelado'),
    ('ORD-007','C006','2024-01-14','entregado'),
    ('ORD-008','C007','2024-01-15','entregado'),
    ('ORD-009','C008','2024-01-16','enviado'),
    ('ORD-010','C002','2024-01-18','entregado'),
    ('ORD-011','C009','2024-01-20','entregado'),
    ('ORD-012','C010','2024-01-22','entregado'),
    ('ORD-013','C003','2024-02-01','entregado'),
    ('ORD-014','C001','2024-02-05','entregado'),
    ('ORD-015','C004','2024-02-08','cancelado'),
    ('ORD-016','C002','2024-02-10','entregado'),
    ('ORD-017','C005','2024-02-12','entregado'),
    ('ORD-018','C006','2024-02-16','entregado'),
    ('ORD-019','C007','2024-02-20','entregado'),
    ('ORD-020','C010','2024-03-12','entregado'),
]

order_items_data = [
    ('ORD-001','P101',1,1200.00,0.10),
    ('ORD-002','P205',2,150.00,0.00),
    ('ORD-003','P310',1,350.00,0.05),
    ('ORD-004','P410',1,45.00,0.00),
    ('ORD-005','P501',3,35.00,0.15),
    ('ORD-006','P101',1,1200.00,0.10),
    ('ORD-007','P602',1,120.00,0.00),
    ('ORD-008','P710',2,80.00,0.05),
    ('ORD-009','P310',2,350.00,0.10),
    ('ORD-010','P801',1,280.00,0.00),
    ('ORD-011','P501',1,35.00,0.00),
    ('ORD-012','P410',3,45.00,0.00),
    ('ORD-013','P602',1,120.00,0.10),
    ('ORD-014','P901',1,500.00,0.05),
    ('ORD-015','P710',1,80.00,0.00),
    ('ORD-016','P801',2,280.00,0.10),
    ('ORD-017','P310',1,350.00,0.00),
    ('ORD-018','P901',1,500.00,0.00),
    ('ORD-019','P101',1,1200.00,0.10),
    ('ORD-020','P901',1,500.00,0.05),
]

cursor.executemany('INSERT INTO customers VALUES (?,?,?,?,?)', customers_data)
cursor.executemany('INSERT INTO products VALUES (?,?,?,?,?)', products_data)
cursor.executemany('INSERT INTO orders VALUES (?,?,?,?)', orders_data)
cursor.executemany('INSERT INTO order_items(order_id,product_id,quantity,unit_price,discount) VALUES (?,?,?,?,?)', order_items_data)
conn.commit()
print('Datos insertados correctamente.')

---
## 2. SELECT básico y filtros (WHERE)

In [ ]:
# Seleccionar todas las columnas de una tabla
query('SELECT * FROM customers')

In [ ]:
# Seleccionar columnas específicas
query('SELECT name, city FROM customers')

In [ ]:
# Filtrar con WHERE
query("SELECT * FROM customers WHERE city = 'Bogotá'")

In [ ]:
# Múltiples condiciones con AND / OR
query("SELECT * FROM products WHERE category = 'Electrónica' AND price < 200")

In [ ]:
# Operador IN
query("SELECT * FROM orders WHERE status IN ('cancelado', 'enviado')")

In [ ]:
# Operador LIKE (buscar patrones en texto)
query("SELECT * FROM customers WHERE name LIKE 'A%'")

In [ ]:
# Operador BETWEEN
query("SELECT * FROM products WHERE price BETWEEN 100 AND 400")

### Ejercicios — Bloque 2
> Escribe la consulta SQL para cada uno de los siguientes ejercicios.

**E2.1** — Lista todos los clientes de Medellín o Cali.

In [ ]:
# Tu respuesta aquí
query("""
-- Escribe tu consulta
SELECT 'Completa este ejercicio' AS tarea
""")

**E2.2** — Lista los productos de la categoría 'Muebles' con precio mayor a 400.

In [ ]:
# Tu respuesta aquí


---
## 3. Ordenamiento (ORDER BY) y límites (LIMIT)

In [ ]:
# Ordenar por precio ascendente
query('SELECT name, price FROM products ORDER BY price ASC')

In [ ]:
# Top 3 productos más caros
query('SELECT name, price FROM products ORDER BY price DESC LIMIT 3')

In [ ]:
# Ordenar por múltiples columnas
query('SELECT name, category, price FROM products ORDER BY category ASC, price DESC')

---
## 4. Funciones de agregación

In [ ]:
# COUNT — contar registros
query('SELECT COUNT(*) AS total_ordenes FROM orders')

In [ ]:
# SUM — sumar valores
query('SELECT SUM(quantity * unit_price) AS ingresos_brutos FROM order_items')

In [ ]:
# AVG, MAX, MIN
query('''
SELECT
    AVG(price) AS precio_promedio,
    MAX(price) AS precio_maximo,
    MIN(price) AS precio_minimo
FROM products
''')

In [ ]:
# Calcular ingresos netos (con descuento)
query('''
SELECT
    ROUND(SUM(quantity * unit_price), 2)                       AS ingresos_brutos,
    ROUND(SUM(quantity * unit_price * discount), 2)            AS total_descuentos,
    ROUND(SUM(quantity * unit_price * (1 - discount)), 2)      AS ingresos_netos
FROM order_items
''')

---
## 5. Agrupación (GROUP BY) y HAVING

In [ ]:
# Ventas por categoría de producto
query('''
SELECT
    p.category,
    COUNT(oi.item_id)                                         AS cantidad_items,
    ROUND(SUM(oi.quantity * oi.unit_price * (1-oi.discount)), 2) AS ingresos_netos
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY ingresos_netos DESC
''')

In [ ]:
# Clientes con más de 1 pedido (HAVING)
query('''
SELECT
    customer_id,
    COUNT(*) AS num_ordenes
FROM orders
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY num_ordenes DESC
''')

In [ ]:
# Ordenes por mes
query('''
SELECT
    SUBSTR(order_date, 1, 7) AS mes,
    COUNT(*) AS total_ordenes,
    COUNT(CASE WHEN status = 'entregado' THEN 1 END) AS entregadas,
    COUNT(CASE WHEN status = 'cancelado' THEN 1 END) AS canceladas
FROM orders
GROUP BY mes
ORDER BY mes
''')

### Ejercicios — Bloque 5

**E5.1** — ¿Cuántos clientes hay por ciudad? Ordena de mayor a menor.

In [ ]:
# Tu respuesta aquí


**E5.2** — Muestra los productos vendidos más de 2 veces (suma de `quantity`).

In [ ]:
# Tu respuesta aquí


---
## 6. JOINs entre tablas

In [ ]:
# INNER JOIN — ordenes con nombre del cliente
query('''
SELECT
    o.order_id,
    c.name AS cliente,
    c.city,
    o.order_date,
    o.status
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_date
''')

In [ ]:
# JOIN de 3 tablas — detalle completo de la orden
query('''
SELECT
    o.order_id,
    c.name       AS cliente,
    p.name       AS producto,
    p.category,
    oi.quantity,
    oi.unit_price,
    oi.discount,
    ROUND(oi.quantity * oi.unit_price * (1-oi.discount), 2) AS total_linea
FROM orders o
JOIN customers c    ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id    = oi.order_id
JOIN products p     ON oi.product_id = p.product_id
ORDER BY o.order_date, o.order_id
''')

In [ ]:
# LEFT JOIN — clientes que NO tienen ordenes
# (En este dataset todos tienen, pero el patrón es importante)
query('''
SELECT
    c.customer_id,
    c.name,
    COUNT(o.order_id) AS total_ordenes
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_ordenes DESC
''')

### Ejercicios — Bloque 6

**E6.1** — Lista las órdenes entregadas con el nombre del cliente y el total de la orden.

In [ ]:
# Tu respuesta aquí


**E6.2** — Para cada producto, muestra cuántas veces fue comprado y el ingreso total que generó.

In [ ]:
# Tu respuesta aquí


---
## 7. Subconsultas

In [ ]:
# Productos con precio mayor al promedio
query('''
SELECT name, price
FROM products
WHERE price > (SELECT AVG(price) FROM products)
ORDER BY price DESC
''')

In [ ]:
# Clientes que hicieron al menos una orden cancelada
query('''
SELECT name, email, city
FROM customers
WHERE customer_id IN (
    SELECT DISTINCT customer_id
    FROM orders
    WHERE status = 'cancelado'
)
''')

In [ ]:
# Subconsulta en FROM (tabla derivada)
query('''
SELECT
    cliente,
    total_gastado,
    CASE
        WHEN total_gastado >= 2000 THEN 'VIP'
        WHEN total_gastado >= 500  THEN 'Regular'
        ELSE 'Ocasional'
    END AS segmento
FROM (
    SELECT
        c.name AS cliente,
        ROUND(SUM(oi.quantity * oi.unit_price * (1-oi.discount)), 2) AS total_gastado
    FROM customers c
    JOIN orders o     ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status = 'entregado'
    GROUP BY c.customer_id
) sub
ORDER BY total_gastado DESC
''')

---
## 8. Funciones de texto y fecha

In [ ]:
# Funciones de texto
query('''
SELECT
    name,
    UPPER(name)            AS nombre_mayus,
    LENGTH(name)           AS longitud,
    SUBSTR(name, 1, 5)     AS primeros_5,
    REPLACE(email, '@email.com', '') AS usuario_email
FROM customers
LIMIT 5
''')

In [ ]:
# Funciones de fecha en SQLite
query('''
SELECT
    order_id,
    order_date,
    SUBSTR(order_date, 1, 4) AS anio,
    SUBSTR(order_date, 6, 2) AS mes,
    SUBSTR(order_date, 9, 2) AS dia
FROM orders
LIMIT 5
''')

---
## 9. Ejercicios integradores

**EI.1** — Ranking de los 5 clientes con mayor gasto total en órdenes entregadas, mostrando: nombre, ciudad, número de órdenes y total gastado.

In [ ]:
# Tu respuesta aquí


**EI.2** — ¿Cuál es el producto más vendido (en unidades) por categoría?

In [ ]:
# Tu respuesta aquí


**EI.3** — Tasa de cancelación por mes: muestra el porcentaje de órdenes canceladas sobre el total de órdenes, por mes.

In [ ]:
# Tu respuesta aquí


**EI.4** — Lista los productos que nunca fueron vendidos (usa LEFT JOIN o NOT IN).

In [ ]:
# Tu respuesta aquí


---
## Soluciones de referencia
Descomenta y ejecuta cada celda para ver la solución.

In [ ]:
# E2.1
# query("SELECT * FROM customers WHERE city IN ('Medellín', 'Cali')")

In [ ]:
# E2.2
# query("SELECT * FROM products WHERE category = 'Muebles' AND price > 400")

In [ ]:
# E5.1
# query('SELECT city, COUNT(*) AS num_clientes FROM customers GROUP BY city ORDER BY num_clientes DESC')

In [ ]:
# E5.2
# query('''
# SELECT p.name, SUM(oi.quantity) AS unidades_vendidas
# FROM order_items oi
# JOIN products p ON oi.product_id = p.product_id
# GROUP BY p.product_id
# HAVING SUM(oi.quantity) > 2
# ORDER BY unidades_vendidas DESC
# ''')

In [ ]:
# EI.1
# query('''
# SELECT c.name, c.city,
#        COUNT(DISTINCT o.order_id) AS num_ordenes,
#        ROUND(SUM(oi.quantity * oi.unit_price * (1-oi.discount)), 2) AS total_gastado
# FROM customers c
# JOIN orders o ON c.customer_id = o.customer_id
# JOIN order_items oi ON o.order_id = oi.order_id
# WHERE o.status = 'entregado'
# GROUP BY c.customer_id
# ORDER BY total_gastado DESC
# LIMIT 5
# ''')

In [ ]:
# Cerrar conexión al finalizar
conn.close()
print('Conexión cerrada.')